Catastrophic Forgetting/Interference

Confirming and Visualizing the interference

In [1]:
import torch
import numpy as np
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset

Setting up Hyperparameters

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
learning_rate = 1e-3
batch_size = 64
epochs_per_task = 3
tasks = [(0, 1), (2, 3), (4, 5), (6, 7), (8, 9)]

In [3]:
print(f"Using device: {device}")

Using device: cpu


Dataset Loading and splitting

In [4]:
mnist_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

In [5]:
train_dataset = datasets.MNIST(root='/home/bukunmi/ml-journey/datasets/data', train=True, download=True, transform=mnist_transform)
test_dataset = datasets.MNIST(root='/home/bukunmi/ml-journey/datasets/data', train=False, download=True, transform=mnist_transform)

In [6]:
# Filter by Digits

def filter_by_digits(dataset, digits):
    indices = [i for i, (_, label) in enumerate(dataset) if label in digits]
    return Subset(dataset, indices)

In [7]:
# Get split MNIST datasets for each task
def get_split_mnist_datasets(tasks_digits):
    train_subset = filter_by_digits(train_dataset, tasks_digits)
    test_subset = filter_by_digits(test_dataset, tasks_digits)
    train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_subset, batch_size=batch_size, shuffle=False)
    return train_loader, test_loader

Instantiate Model, optimizer, criterion and task loaders

In [8]:
task_loaders = [get_split_mnist_datasets(task) for task in tasks]

#### PROGRESSIVE LAYER

In [9]:
class P_Layer(nn.Module):
    def __init__(self, fan_in, fan_out, active_idx):
        self.active_idx = active_idx
        self.W = nn.Linear(fan_in, fan_out)
        self.V = nn.ModuleList([nn.Linear(fan_in, fan_out) for _ in range(active_idx)])
    # prev_layers_outputs
    def forward(self, plo):
        out = self.W(plo[self.active_idx])
        for j in range(self.active_idx):
            out += self.V[j](plo)
        return F.relu(out)

In [ ]:
class P_Net(nn.Module):
    def __init__(self, layer_sizes=[784, 128, 64, 2]):
        super().__init__()
        self.layer_sizes = layer_sizes
        self.columns = nn.ModuleList([])

    def add_columns(self):
        new_idx = len(self.columns)
        for param in self.parameters():
            param.requires_grad = False
        
        self.column_layers = nn.ModuleList()
        for i in range(len(self.layer_sizes) - 1):
            if i < len(self.layer_sizes) - 2:
                self.column_layers.append(P_Layer(self.layer_sizes[i], self.layer_sizes[i+1], new_idx))